[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbanuelos/grad_math_modeling/blob/main/Lectures/Module0_TimeSeries/04_AutoregressiveModels.ipynb)

# Autoregressive Models: Intuition and Stability

**Module 0 · Lesson 4 of 13 · Student edition**  
**Estimated class time:** 80–95 minutes  
**Source sequence:** Original Day 2  

**Prerequisite:** Lessons 1–3  

## Learning objectives

By the end of this lesson, you should be able to:

- Simulate AR(1) and AR(2) processes.
- Explain how autoregressive coefficients control memory and stability.
- Connect AR order to ACF and PACF signatures.

## Before We Begin

> **Prompt:** Think about something you do on autopilot — a habit, a routine, a reflex.
> How much does your *past behavior* predict your *next action*?
>
> Write 3–5 sentences describing this habit. Then draw a simple diagram showing how one moment leads to the next.


---
### Quick Recap from Day 1

Yesterday we learned to *diagnose* a time series: check for stationarity, apply
transformations, and read ACF/PACF plots.

Today we ask: **how do we actually model the dependence structure we found?**

In this notebook you will:
1. Understand what an autoregressive (AR) model is and how it works
2. Simulate AR(1) processes and observe how the parameter $\phi$ controls behavior
3. Derive and apply the stability (stationarity) condition for AR(1)
4. Fit an AR model to real data using `statsmodels`
5. Re-frame a time series as a supervised learning problem using **lag-embedded feature matrices**
6. Build lag matrices for both univariate and multivariate time series


---
## Part 0 — Imports & Setup

Run the cell below. You do **not** need to modify it.


In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

# Time series tools
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

# Reproducibility
np.random.seed(42)

# Display settings
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
%matplotlib inline

print('All imports successful!')


---
## Part 1 — What is an Autoregressive Model?

### 1.1 The AR(1) model

An **autoregressive model of order 1**, written **AR(1)**, says that the current value
of a series depends linearly on the *previous* value plus some random noise:

$$x_t = \phi \, x_{t-1} + \varepsilon_t$$

where:
- $x_t$ is the value at time $t$
- $\phi$ (phi) is the **autoregressive coefficient** — it controls how strongly the past
  influences the present
- $\varepsilon_t \sim \mathcal{N}(0, \sigma^2)$ is white noise (random, mean-zero error)

More generally, an **AR(p)** model uses the last $p$ values:

$$x_t = \phi_1 x_{t-1} + \phi_2 x_{t-2} + \cdots + \phi_p x_{t-p} + \varepsilon_t$$

### 1.2 Simulating AR(1) processes

Before fitting a model to real data, it helps to *simulate* the process and see how
different values of $\phi$ produce very different behaviors.

We will complete the cell below together.


In [ ]:
def simulate_ar1(phi, n=200, sigma=1.0, x0=0.0):
    """Simulate an AR(1) process: x[t] = phi * x[t-1] + noise."""
    x = np.zeros(n)
    x[0] = x0
    for t in range(1, n):
        x[t] = ???
    return x

phi_values = [0.0, 0.5, 0.9, 0.99, 1.0, 1.05, -0.7]
labels     = [
    'phi=0.0 (white noise)',
    'phi=0.5 (moderate memory)',
    'phi=0.9 (strong memory)',
    'phi=0.99 (near unit root)',
    'phi=1.0 (random walk)',
    'phi=1.05 (explosive)',
    'phi=-0.7 (oscillating)',
]

# create a for-loop to plot the different values of phi

### ✏️ Written Response 1.2

Study the seven simulated series above and answer:

1. What happens to the series as $\phi$ increases from 0 toward 1? Describe the change in
   behavior in your own words.
2. What is qualitatively different about $\phi = 1.0$ versus $\phi = 0.99$?
3. What does $\phi = 1.05$ do? Why might this be problematic for a model?
4. What does a **negative** $\phi$ produce? Why does the series oscillate?

> **YOUR ANSWER:**


---
## Part 2 — The Stability Condition for AR(1)

### 2.1 Why does $|\phi| < 1$ matter?

From your simulations above, you can see that the AR(1) process behaves very differently
depending on $\phi$. The **stability condition** for an AR(1) process is:

$$|\phi| < 1$$

When this holds, the process is **stationary** — it has a well-defined, constant mean and
variance over time. When $|\phi| \geq 1$, the process is non-stationary (it either
drifts like a random walk or explodes).

### 2.2 Deriving the mean and variance of a stationary AR(1)

If $x_t = \phi x_{t-1} + \varepsilon_t$ is stationary, then by definition the mean
$\mu = \mathbb{E}[x_t]$ does not change over time:

$$\mu = \phi \mu + 0 \implies \mu(1 - \phi) = 0 \implies \mu = 0$$

(assuming zero-mean noise; if there's a constant term $c$, then $\mu = c / (1-\phi)$)

Similarly, the variance $\gamma_0 = \text{Var}(x_t)$ satisfies:

$$\gamma_0 = \frac{\sigma^2}{1 - \phi^2}$$

Notice: this only makes sense when $|\phi| < 1$. Otherwise, what happens?

### 2.3 Verify with simulation

**Your turn!** Fill in the cell below to compute the **theoretical variance** and compare
it to the **sample variance** from a simulation.


In [ ]:
phi   = 0.7
sigma = 1.0
n     = 5000

# FILL IN: theoretical variance for a stationary AR(1)
theoretical_var = ???

# Simulate and compute sample variance


print(f'phi             = {phi}')
print(f'Theoretical var = {theoretical_var:.4f}')
print(f'Sample var      = {sample_var:.4f}')
print(f'Difference      = {abs(theoretical_var - sample_var):.4f}')


### ✏️ Written Response 2.3

1. How close is the sample variance to the theoretical variance? What does this confirm?
2. **dividir y confluir:** Try changing `phi` to a different value (groups of 2 to 3). What happens to the theoretical variance? Why?
3. What happens if you try `phi = 1.0`? (what error or result do you get,
   and why does it make mathematical sense?)

> **YOUR ANSWER:**


---
## Part 3 — ACF Signature of AR Models

From Day 1, recall:
- **ACF** decays slowly for non-stationary series
- **PACF** cuts off at lag $p$ for an AR($p$) process

Let's verify this for AR(1) and AR(2) simulations.

### 3.1 ACF and PACF of simulated AR processes

Complete the cell below, then run it and study how the PACF behaves.


In [ ]:
def simulate_ar2(phi1, phi2, n=500, sigma=1.0):
    """Simulate an AR(2) process: x[t] = phi1*x[t-1] + phi2*x[t-2] + noise."""
    x = np.zeros(n)
    for t in range(2, n):
        x[t] = ???
    return x

ar1_series = simulate_ar1(phi=0.8, n=500)
ar2_series = simulate_ar2(phi1=0.6, phi2=0.3, n=500)

fig, axes = plt.subplots(2, 2, figsize=(14, 7))

plot_acf( ar1_series, lags=20, ax=axes[0, 0])
axes[0, 0].set_title('ACF — AR(1), phi=0.8')

plot_pacf(ar1_series, lags=20, ax=axes[0, 1], method='ywm')
axes[0, 1].set_title('PACF — AR(1), phi=0.8')

plot_acf( ar2_series, lags=20, ax=axes[1, 0])
axes[1, 0].set_title('ACF — AR(2), phi1=0.6, phi2=0.3')

plot_pacf(ar2_series, lags=20, ax=axes[1, 1], method='ywm')
axes[1, 1].set_title('PACF — AR(2), phi1=0.6, phi2=0.3')

plt.tight_layout()
plt.show()


### ✏️ Written Response 3.1

1. In the AR(1) PACF, how many lags are significant (outside the blue band)?
   What does this tell you about the order of the process?
2. In the AR(2) PACF, how many lags are significant? How does this differ from AR(1)?
3. The ACF for both processes shows a gradual decay. Why isn't the ACF as useful as
   the PACF for identifying the *order* of an AR model?

> **YOUR ANSWER:**
